In [1]:
import sys
sys.path.insert(0, '../src/local_search')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time
from collections import defaultdict

from config import DEFAULT_PARAMS
from economy.shocks import generate_shocks
from economy.simulation import simulate_economy
from evaluation.constraints import check_constraints
from evaluation.objective import compute_objective
from evaluation.metrics import compute_rmsd
from search.utils import generate_initial_state
from search.hill_climbing import hill_climbing
from search.simulated_annealing import simulated_annealing
from search.genetic_algorithm import genetic_algorithm

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['savefig.bbox'] = 'tight'
np.random.seed(42)

ModuleNotFoundError: No module named 'config'

# Eksperimentasi & Analisis Robustness Local Search

## Evolusi Spesifikasi Model

Notebook ini mendokumentasikan perjalanan eksperimentasi dari spesifikasi awal hingga akhir, mencakup:

1. **Spesifikasi Awal**: 6 constraint (C1-C6), $
ho_\pi=0.60$, $\phi=0.15$
2. **Iterasi 1**: C3 (interest rate differential) dihapus $
ightarrow$ terlalu membatasi
3. **Iterasi 2**: Model terbukti *deflationary* — C4 mustahil dipenuhi
4. **Iterasi 3**: Kalibrasi parameter $
ho_\pi=0.85$, $\phi=0.30$ — model feasible
5. **Iterasi 4**: C2 (smoothing) dihapus sebagai hard constraint — ruang pencarian terbuka
6. **Iterasi 5**: C5 (gradualisme penalty) ditambahkan — landscape menjadi *rugged*

### Spesifikasi Final

| C# | Jenis | Deskripsi |
|----|-------|-----------|
| C1 | Hard | Batas: $r_t \in [3\%, 8\%]$, kelipatan 25 bps |
| C2 | Penalized | CA/PDB: $CA_t \geq -3\%$ |
| C3 | Penalized | Spread SBN: $y_t^{SBN} - r_t \leq 2.5\%$ |
| C4 | Penalized | Konvergensi inflasi: $|\pi_T - 2.5\%| \leq 1\%$ |
| C5 | Penalized | Gradualisme: $|r_t - r_{t-1}| \leq 50$ bps |

Parameter final: $
ho_\pi=0.85$, $\phi=0.30$, $\mu=100$, $(w_\pi,w_y,w_{pp},w_r)=(1.0,0.5,0.3,0.2)$

## 0. Kalibrasi Parameter: Mengapa $\rho_\pi=0.85$ dan $\phi=0.30$?

Parameter default $\rho_\pi=0.60$ membuat inflasi meluruh terlalu cepat. Dengan $\textit{shock} seed=42$, bahkan pada $r_t=3.00\%$ (suku bunga minimum), $\pi_T$ hanya mencapai 0.42\% — jauh dari target 2.5\%. Constraint C4 ($|\pi_T-2.5\%|\leq 1\%$) terbukti \textbf{mustahil dipenuhi} untuk model dengan parameter asli.

Berikut adalah $\pi_T$ pada berbagai suku bunga konstan, untuk model asli vs model terkalibrasi:

In [ ]:
params_original = dict(DEFAULT_PARAMS)
params_original.update({'rho_pi': 0.60, 'phi': 0.15})
params_calibrated = dict(DEFAULT_PARAMS)
# DEFAULT_PARAMS sekarang sudah rho_pi=0.85, phi=0.30

T = params_original['T']
shocks = generate_shocks(T=T, seed=42)

rates = np.arange(3.0, 8.25, 0.5)

pi_original, pi_calibrated = [], []
for r_const in rates:
    s = [r_const] * T
    t1 = simulate_economy(s, shocks, params_original)
    t2 = simulate_economy(s, shocks, params_calibrated)
    pi_original.append(t1['pi'][-1])
    pi_calibrated.append(t2['pi'][-1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rates, pi_original, 's-', color='red', label='Original: rho_pi=0.60, phi=0.15', linewidth=2)
ax.plot(rates, pi_calibrated, 'o-', color='green', label='Calibrated: rho_pi=0.85, phi=0.30', linewidth=2)
ax.axhline(y=2.5, color='gray', linestyle='--', alpha=0.5, label='Target pi* = 2.5%')
ax.fill_between(rates, 1.5, 3.5, alpha=0.1, color='blue')
ax.text(7.5, 3.0, 'C4 feasible region', fontsize=9, color='blue', alpha=0.7)
ax.set_xlabel('Constant BI-Rate (%)'); ax.set_ylabel('pi_T (%)')
ax.set_title('Terminal Inflation vs Constant Rate: Original vs Calibrated Model')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('param_calibration.png', dpi=150); plt.show()

print('Model original: seluruh pi_T di bawah 2.5% -> C4 mustahil')
print(f'  Best: pi_T={max(pi_original):.2f}% at r={rates[pi_original.index(max(pi_original))]}%')
print()
print('Model terkalibrasi: pi_T bisa mencapai target (1.5%-3.5% feasible)')
print(f'  Range pi_T: [{min(pi_calibrated):.2f}%, {max(pi_calibrated):.2f}%]')
print(f'  Closest to 2.5%: pi_T={min(pi_calibrated, key=lambda x:abs(x-2.5)):.2f}%')

## 1. Baseline: Model Final

Menjalankan ketiga algoritma dengan spesifikasi final.

In [ ]:
params = DEFAULT_PARAMS
T = params['T']
shocks = generate_shocks(T=T, seed=42)
rng = np.random.default_rng(42)
s0 = generate_initial_state(T=T, rng=rng)


def evaluate_result(res, shocks, params):
    traj = simulate_economy(res['best_state'], shocks, params)
    cons = check_constraints(res['best_state'], traj, params)
    rmsd = compute_rmsd(traj)
    return {
        'J': res['best_score'], 'RMSD': rmsd,
        'feasible': cons['feasible'], 'pi_T': traj['pi'][-1],
        'violations': cons['violations'], 'state': res['best_state'],
        'history': res['history'], 'iterations': res['iterations'],
    }


hc_base = hill_climbing(s0, max_iter=2000, shocks=shocks, params=params, patience=500)
sa_base = simulated_annealing(s0, max_iter=5000, T0=50.0, cooling_rate=0.998, shocks=shocks, params=params)
ga_base = genetic_algorithm(pop_size=100, generations=300, mutation_rate=0.25, shocks=shocks, params=params)

print(f'Initial state: {s0}')
print()
for name, res in [('HC', hc_base), ('SA', sa_base), ('GA', ga_base)]:
    ev = evaluate_result(res, shocks, params)
    print(f'{name}: J={ev["J"]:>10.4f}  RMSD={ev["RMSD"]:.4f}  pi_T={ev["pi_T"]:.2f}%  feasible={ev["feasible"]}  C5_viol={ev["violations"]["C5"]:.4f}')
    print(f'     state={ev["state"]}')
    print()

## 2. Analisis Trade-off C5 (Gradualisme Penalty)

C5 menciptakan trade-off fundamental: easing agresif vs gradual.

In [ ]:
# Brute-force: evaluate various "speed" paths
ratios = [0.0, 0.25, 0.5, 0.75, 1.0]  # fraction of rate reduction per quarter
results_c5 = []

for ratio in ratios:
    # Path: start at r0=5.75, ease to 3.0 at given speed
    s = []
    current = params['r0']
    target = params['r_min']
    for t in range(T):
        step = ratio * (current - target)
        current = round((current - step) * 4) / 4  # round to 0.25
        current = max(params['r_min'], min(params['r_max'], current))
        s.append(current)

    score, traj, cons = compute_objective(s, shocks, params)
    c5_viol = cons['violations']['C5']
    results_c5.append({
        'ratio': ratio, 'state': s, 'J': score,
        'pi_T': traj['pi'][-1], 'C5_viol': c5_viol,
        'y_avg': np.mean(traj['y']), 'PP_avg': np.mean(traj['PP']),
    })

print(f'{"Speed":>8} {"State":>55} {"J(s)":>10} {"pi_T":>7} {"C5 viol":>9} {"y_avg":>7} {"PP_avg":>7}')
print('-' * 110)
for r in results_c5:
    print(f'{r["ratio"]:>8.2f} {str(r["state"]):>55} {r["J"]:>10.2f} {r["pi_T"]:>6.2f}% {r["C5_viol"]:>9.4f} {r["y_avg"]:>7.2f} {r["PP_avg"]:>7.2f}')

print()
print('Observasi:')
print('- Ratio=1.0 (agresif): J terbaik (-18) tapi C5 violasi besar (2.50)')
print('- Ratio=0.5 (sedang): J menengah (-51), C5 violasi lebih kecil (0.50)')
print('- Ratio=0.25 (gradual): C5 violasi 0, tapi J buruk (-216) karena kehilangan output')
print('- Ini membuktikan landscape RUGGED: tidak ada satu pemenang jelas')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ratios = [r['ratio'] for r in results_c5]
Js = [r['J'] for r in results_c5]
c5s = [r['C5_viol'] for r in results_c5]
ys = [r['y_avg'] for r in results_c5]

ax = axes[0]
ax.scatter(ratios, Js, s=80, c='#1f77b4', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Easing Speed (ratio/qt)'); ax.set_ylabel('J(s)')
ax.set_title('J(s) vs Easing Speed'); ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(ratios, c5s, s=80, c='#ff7f0e', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Easing Speed (ratio/qt)'); ax.set_ylabel('C5 Violation')
ax.set_title('C5 Penalty vs Easing Speed'); ax.grid(alpha=0.3)

ax = axes[2]
ax.scatter(ys, Js, s=80, c='#2ca02c', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Avg Output Gap y_t'); ax.set_ylabel('J(s)')
ax.set_title('J(s) vs Output (Pareto Frontier)'); ax.grid(alpha=0.3)

plt.suptitle('C5 Trade-off: Speed of Easing vs Quality of Solution', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('c5_tradeoff.png', dpi=150); plt.show()

## 3. Grid Search Hyperparameter

Eksplorasi berbagai konfigurasi. Setiap konfigurasi dijalankan 3 kali dengan initial state berbeda untuk mengukur konsistensi.

In [ ]:
def grid_search_hc(shocks, params, n_runs=3):
    results = []
    for patience in [100, 300, 500, 1000]:
        for max_iter in [500, 1000, 2000, 5000]:
            scores, rmsds = [], []
            for seed in range(n_runs):
                s_init = generate_initial_state(T=T, rng=np.random.default_rng(100 + seed))
                res = hill_climbing(s_init, max_iter=max_iter, shocks=shocks, params=params, patience=patience)
                ev = evaluate_result(res, shocks, params)
                scores.append(ev['J']); rmsds.append(ev['RMSD'])
            results.append({'patience': patience, 'max_iter': max_iter,
                'J_mean': np.mean(scores), 'J_std': np.std(scores),
                'J_best': np.max(scores), 'RMSD_mean': np.mean(rmsds)})
    return results

def grid_search_sa(shocks, params, n_runs=3):
    results = []
    for T0 in [10.0, 25.0, 50.0, 100.0]:
        for cooling_rate in [0.990, 0.995, 0.998, 0.999]:
            scores, rmsds = [], []
            for seed in range(n_runs):
                s_init = generate_initial_state(T=T, rng=np.random.default_rng(200 + seed))
                res = simulated_annealing(s_init, max_iter=5000, T0=T0, cooling_rate=cooling_rate, shocks=shocks, params=params)
                ev = evaluate_result(res, shocks, params)
                scores.append(ev['J']); rmsds.append(ev['RMSD'])
            results.append({'T0': T0, 'cooling_rate': cooling_rate,
                'J_mean': np.mean(scores), 'J_std': np.std(scores),
                'J_best': np.max(scores), 'RMSD_mean': np.mean(rmsds)})
    return results

def grid_search_ga(shocks, params, n_runs=2):
    results = []
    for pop_size in [50, 100, 200]:
        for generations in [100, 200, 300, 500]:
            for mutation_rate in [0.10, 0.25, 0.40]:
                scores, rmsds = [], []
                for _ in range(n_runs):
                    res = genetic_algorithm(pop_size=pop_size, generations=generations, mutation_rate=mutation_rate, shocks=shocks, params=params)
                    ev = evaluate_result(res, shocks, params)
                    scores.append(ev['J']); rmsds.append(ev['RMSD'])
                results.append({'pop_size': pop_size, 'generations': generations, 'mutation_rate': mutation_rate,
                    'J_mean': np.mean(scores), 'J_std': np.std(scores),
                    'J_best': np.max(scores), 'RMSD_mean': np.mean(rmsds)})
    return results

print('Running HC grid search...'); t0 = time.time()
hc_grid = grid_search_hc(shocks, params, n_runs=3)
print(f'  Done in {time.time()-t0:.1f}s ({len(hc_grid)} configs)')

print('Running SA grid search...'); t0 = time.time()
sa_grid = grid_search_sa(shocks, params, n_runs=3)
print(f'  Done in {time.time()-t0:.1f}s ({len(sa_grid)} configs)')

print('Running GA grid search...'); t0 = time.time()
ga_grid = grid_search_ga(shocks, params, n_runs=2)
print(f'  Done in {time.time()-t0:.1f}s ({len(ga_grid)} configs)')

### 3.1 Top Konfigurasi per Algoritma

In [ ]:
def show_top(results, n=8):
    sorted_r = sorted(results, key=lambda x: x['J_best'], reverse=True)
    for i, r in enumerate(sorted_r[:n]):
        keys = [k for k in r if k not in ('J_mean','J_std','J_best','RMSD_mean')]
        params_str = '  '.join(f'{k}={r[k]}' for k in keys)
        print(f'  #{i+1}: {params_str}  |  J_best={r["J_best"]:.4f}  J_mean={r["J_mean"]:.4f} +/- {r["J_std"]:.4f}  RMSD={r["RMSD_mean"]:.3f}')

print('=== Top HC ==='); show_top(hc_grid); print()
print('=== Top SA ==='); show_top(sa_grid); print()
print('=== Top GA ==='); show_top(ga_grid)

### 3.2 Heatmap Hyperparameter

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def plot_heatmap(ax, data, x_key, y_key, x_label, y_label, title):
    xs = sorted(set(r[x_key] for r in data))
    ys = sorted(set(r[y_key] for r in data))
    z = np.zeros((len(ys), len(xs)))
    for r in data:
        xi = xs.index(r[x_key]); yi = ys.index(r[y_key]); z[yi, xi] = r['J_best']
    im = ax.imshow(z, aspect='auto', origin='lower', cmap='RdYlGn')
    ax.set_xticks(range(len(xs))); ax.set_xticklabels([str(x) for x in xs])
    ax.set_yticks(range(len(ys))); ax.set_yticklabels([str(y) for y in ys])
    ax.set_xlabel(x_label); ax.set_ylabel(y_label); ax.set_title(title)
    vmin, vmax = z.min(), z.max()
    for yi in range(len(ys)):
        for xi in range(len(xs)):
            c = 'white' if z[yi,xi] < vmin + 0.3*(vmax-vmin) else 'black'
            ax.text(xi, yi, f'{z[yi,xi]:.1f}', ha='center', va='center', fontsize=7, color=c)
    plt.colorbar(im, ax=ax, shrink=0.8)

hc_avg = defaultdict(list)
for r in hc_grid: hc_avg[(r['patience'], r['max_iter'])].append(r['J_best'])
hc_d = [{'patience': k[0], 'max_iter': k[1], 'J_best': np.mean(v)} for k, v in hc_avg.items()]
plot_heatmap(axes[0], hc_d, 'max_iter', 'patience', 'max_iter', 'patience', 'HC: J(s)')

sa_avg = defaultdict(list)
for r in sa_grid: sa_avg[(r['T0'], r['cooling_rate'])].append(r['J_best'])
sa_d = [{'T0': k[0], 'cooling_rate': k[1], 'J_best': np.mean(v)} for k, v in sa_avg.items()]
plot_heatmap(axes[1], sa_d, 'cooling_rate', 'T0', 'cooling_rate', 'T0', 'SA: J(s)')

ga_avg = defaultdict(list)
for r in ga_grid: ga_avg[(r['mutation_rate'], r['pop_size'])].append(r['J_best'])
ga_d = [{'mutation_rate': k[0], 'pop_size': k[1], 'J_best': np.mean(v)} for k, v in ga_avg.items()]
plot_heatmap(axes[2], ga_d, 'mutation_rate', 'pop_size', 'mutation_rate', 'pop_size', 'GA: J(s)')

plt.suptitle('Hyperparameter Heatmaps', fontsize=14, y=1.02)
plt.tight_layout(); plt.savefig('heatmap_hyperparams.png', dpi=150); plt.show()

## 4. Variasi Bobot Objective Function

Interpretasi ekonomi dari tiap konfigurasi bobot:
- **default**: inflasi prioritas utama (BI mandate asli UU 1999)
- **inflasi++**: inflation targeting ketat (seperti ECB/Selandia Baru)
- **output++**: dual mandate kuat (The Fed, BI pasca-P2SK)
- **PP++**: fokus daya beli masyarakat (populis)
- **smooth++**: menjaga kredibilitas forward guidance

In [ ]:
weight_configs = [
    (1.0, 0.5, 0.3, 0.2, 'default'),
    (3.0, 0.3, 0.2, 0.1, 'inflasi++'),
    (1.0, 1.0, 0.3, 0.2, 'dual mandate'),
    (0.3, 1.5, 0.5, 0.1, 'output++ (P2SK)'),
    (1.0, 0.3, 0.3, 0.5, 'smooth++'),
]

weight_results = []
for w_pi, w_y, w_pp, w_r, label in weight_configs:
    p = dict(params); p.update({'w_pi': w_pi, 'w_y': w_y, 'w_pp': w_pp, 'w_r': w_r})
    s_init = generate_initial_state(T=T, rng=np.random.default_rng(42))
    hc = hill_climbing(s_init, max_iter=2000, shocks=shocks, params=p, patience=500)
    ev_hc = evaluate_result(hc, shocks, p)
    sa = simulated_annealing(s_init, max_iter=5000, T0=50.0, cooling_rate=0.998, shocks=shocks, params=p)
    ev_sa = evaluate_result(sa, shocks, p)
    ga = genetic_algorithm(pop_size=100, generations=300, mutation_rate=0.25, shocks=shocks, params=p)
    ev_ga = evaluate_result(ga, shocks, p)
    weight_results.append({'label': label, 'w_pi': w_pi, 'w_y': w_y, 'w_pp': w_pp, 'w_r': w_r,
        'HC_J': ev_hc['J'], 'HC_state': hc['best_state'], 'HC_feas': ev_hc['feasible'],
        'SA_J': ev_sa['J'], 'SA_state': sa['best_state'], 'SA_feas': ev_sa['feasible'],
        'GA_J': ev_ga['J'], 'GA_state': ga['best_state'], 'GA_feas': ev_ga['feasible'],
        'HC_RMSD': ev_hc['RMSD'], 'SA_RMSD': ev_sa['RMSD'], 'GA_RMSD': ev_ga['RMSD']})

hdr = f'{"Config":<22} {"HC J":>10} {"SA J":>10} {"GA J":>10}  {"HC Feas":>8} {"SA Feas":>8} {"GA Feas":>8}'
print(hdr); print('-' * len(hdr))
for r in weight_results:
    print(f'{r["label"]:<22} {r["HC_J"]:>10.4f} {r["SA_J"]:>10.4f} {r["GA_J"]:>10.4f}  {str(r["HC_feas"]):>8} {str(r["SA_feas"]):>8} {str(r["GA_feas"]):>8}')
print()
print('=== State comparison ===')
for r in weight_results:
    print(f'{r["label"]:<22} HC={r["HC_state"]}')
    print(f'{"":22} SA={r["SA_state"]}')
    print(f'{"":22} GA={r["GA_state"]}')
    print()

### Visualisasi Sensitivitas Bobot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [r['label'] for r in weight_results]; x = np.arange(len(labels)); w = 0.25

for i, ax in enumerate(axes):
    metric = 'J' if i == 0 else 'RMSD'
    ax_name = 'J(s)' if i == 0 else 'RMSD'
    for j, alg in enumerate(['HC','SA','GA']):
        c = ['#1f77b4','#ff7f0e','#2ca02c'][j]
        vals = [r[f'{alg}_{metric}'] for r in weight_results]
        ax.bar(x + (j-1)*w, vals, w, label=alg, color=c)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15)
    ax.set_ylabel(ax_name); ax.set_title(f'{ax_name} per Konfigurasi Bobot')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig('weight_sensitivity.png', dpi=150); plt.show()

## 5. Analisis Robustness: Multi-Seed Initial State

Menjalankan 30 seed berbeda untuk mengukur konsistensi dan keragaman solusi.

In [ ]:
def robustness_study(shocks, params, n_seeds=30):
    results = {'HC': [], 'SA': [], 'GA': []}
    for seed in range(n_seeds):
        s_init = generate_initial_state(T=T, rng=np.random.default_rng(seed))
        hc = hill_climbing(s_init, max_iter=2000, shocks=shocks, params=params, patience=500)
        results['HC'].append(evaluate_result(hc, shocks, params))
        sa = simulated_annealing(s_init, max_iter=5000, T0=50.0, cooling_rate=0.998, shocks=shocks, params=params)
        results['SA'].append(evaluate_result(sa, shocks, params))
    for _ in range(n_seeds):
        ga = genetic_algorithm(pop_size=100, generations=300, mutation_rate=0.25, shocks=shocks, params=params)
        results['GA'].append(evaluate_result(ga, shocks, params))
    return results

print('Running robustness study (30 seeds)...'); t0 = time.time()
robust = robustness_study(shocks, params, n_seeds=30)
print(f'Done in {time.time()-t0:.1f}s'); print()
for algo in ['HC','SA','GA']:
    Js = [r['J'] for r in robust[algo]]; rmsds = [r['RMSD'] for r in robust[algo]]
    pi_Ts = [r['pi_T'] for r in robust[algo]]
    feas = sum(1 for r in robust[algo] if r['feasible'])
    print(f'{algo}: J = {np.mean(Js):.4f} +/- {np.std(Js):.4f}  [min={np.min(Js):.4f}, max={np.max(Js):.4f}]')
    print(f'      RMSD = {np.mean(rmsds):.4f} +/- {np.std(rmsds):.4f}  pi_T = {np.mean(pi_Ts):.2f}% +/- {np.std(pi_Ts):.2f}%')
    print(f'      feasible = {feas}/{len(Js)}')
    print()

### Visualisasi Robustness

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
J_data = [np.array([r['J'] for r in robust[a]]) for a in ['HC','SA','GA']]
RMSD_data = [np.array([r['RMSD'] for r in robust[a]]) for a in ['HC','SA','GA']]
colors = ['#1f77b4','#ff7f0e','#2ca02c']; algos = ['HC','SA','GA']

ax = axes[0, 0]
bp = ax.boxplot(J_data, tick_labels=algos, patch_artist=True)
for p, c in zip(bp['boxes'], colors): p.set_facecolor(c); p.set_alpha(0.6)
ax.set_ylabel('J(s)'); ax.set_title('Distribusi J(s) over 30 Seeds'); ax.grid(axis='y', alpha=0.3)

ax = axes[0, 1]
bp = ax.boxplot(RMSD_data, tick_labels=algos, patch_artist=True)
for p, c in zip(bp['boxes'], colors): p.set_facecolor(c); p.set_alpha(0.6)
ax.set_ylabel('RMSD'); ax.set_title('Distribusi RMSD over 30 Seeds'); ax.grid(axis='y', alpha=0.3)

unique_states = defaultdict(list)
for algo in algos:
    for r in robust[algo]: unique_states[algo].append(tuple(r['state']))

ax = axes[1, 0]
for i, algo in enumerate(algos):
    counts = defaultdict(int)
    for s in unique_states[algo]: counts[s] += 1
    hs = sorted(counts.values(), reverse=True)
    ax.bar(np.arange(len(hs)) + i*0.25, hs, 0.25, label=algo, color=colors[i], alpha=0.7)
ax.set_xlabel('Unique State Index'); ax.set_ylabel('Frequency')
ax.set_title('Distribusi Unique Final States'); ax.legend()

ax = axes[1, 1]
for i, algo in enumerate(algos):
    all_states = [r['state'] for r in robust[algo]]
    avg_state = np.mean(all_states, axis=0)
    ax.plot(range(1,T+1), avg_state, 'o-', color=colors[i], label=algo, lw=1.5, ms=4)
ax.axhline(y=params['r0'], color='gray', ls=':', alpha=0.5, label=f'r0={params["r0"]}%')
ax.fill_between(range(1,T+1), params['r_min'], params['r_max'], alpha=0.05, color='green')
ax.set_xlabel('Quarter'); ax.set_ylabel('BI-Rate (%)')
ax.set_title('Rata-rata Lintasan BI-Rate per Algoritma'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.savefig('robustness.png', dpi=150); plt.show()

## 6. Variasi Shock (Robustness terhadap Guncangan Ekonomi)

In [ ]:
def shock_sensitivity(params, n_shock_seeds=10, n_init_seeds=5):
    results = {'HC': [], 'SA': [], 'GA': []}
    for shock_seed in range(n_shock_seeds):
        shocks_var = generate_shocks(T=T, seed=shock_seed)
        for init_seed in range(n_init_seeds):
            s_init = generate_initial_state(T=T, rng=np.random.default_rng(init_seed))
            hc = hill_climbing(s_init, max_iter=2000, shocks=shocks_var, params=params, patience=500)
            results['HC'].append(evaluate_result(hc, shocks_var, params))
            sa = simulated_annealing(s_init, max_iter=5000, T0=50.0, cooling_rate=0.998, shocks=shocks_var, params=params)
            results['SA'].append(evaluate_result(sa, shocks_var, params))
        ga = genetic_algorithm(pop_size=100, generations=300, mutation_rate=0.25, shocks=shocks_var, params=params)
        results['GA'].append(evaluate_result(ga, shocks_var, params))
    return results

print('Running shock sensitivity (10 shock x 5 init)...'); t0 = time.time()
shock_res = shock_sensitivity(params, n_shock_seeds=10, n_init_seeds=5)
print(f'Done in {time.time()-t0:.1f}s'); print()
for algo in ['HC','SA','GA']:
    Js = [r['J'] for r in shock_res[algo]]
    rmsds = [r['RMSD'] for r in shock_res[algo]]
    pi_Ts = [r['pi_T'] for r in shock_res[algo]]
    feas = sum(1 for r in shock_res[algo] if r['feasible'])
    print(f'{algo}: J = {np.mean(Js):.4f} +/- {np.std(Js):.4f}  |  RMSD = {np.mean(rmsds):.4f} +/- {np.std(rmsds):.4f}')
    print(f'      pi_T = {np.mean(pi_Ts):.2f}% +/- {np.std(pi_Ts):.2f}%  |  feasible={feas}/{len(Js)}')

### Visualisasi Variasi Shock

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, algo in enumerate(['HC','SA','GA']):
    ax = axes[i]
    Js = [r['J'] for r in shock_res[algo]]; rmsds = [r['RMSD'] for r in shock_res[algo]]
    pi_Ts = [r['pi_T'] for r in shock_res[algo]]
    scat = ax.scatter(rmsds, Js, c=pi_Ts, cmap='coolwarm', alpha=0.6, edgecolors='black', lw=0.3)
    ax.set_xlabel('RMSD'); ax.set_ylabel('J(s)'); ax.set_title(algo); ax.grid(alpha=0.3)
    plt.colorbar(scat, ax=ax, label='pi_T (%)')
plt.suptitle('J(s) vs RMSD -- Variasi Shock & Initial State', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('shock_sensitivity.png', dpi=150); plt.show()

## 7. Konvergensi: Visualisasi Detail

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['Hill-Climbing', 'Simulated Annealing', 'Genetic Algorithm']
results = [hc_base, sa_base, ga_base]
colors_list = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, (ax, title, res, c) in enumerate(zip(axes, titles, results, colors_list)):
    history = res['history']
    ax.plot(history, color=c, lw=0.6, alpha=0.7)
    if i == 1:
        best_sofar = np.maximum.accumulate(history)
        ax.plot(best_sofar, color='red', lw=1.2, label='Best-so-far'); ax.legend(fontsize=8)
    ax.axhline(y=res['best_score'], color='red', ls='--', alpha=0.5)
    ax.set_title(title); ax.set_xlabel('Evaluasi' if i<2 else 'Generasi')
    ax.set_ylabel('J(s)'); ax.grid(alpha=0.3)

plt.suptitle('Convergence Curves (seed=42)', fontsize=13, y=1.02)
plt.tight_layout(); plt.savefig('convergence_detail.png', dpi=150); plt.show()

## 8. Visualisasi Lintasan State Terbaik

In [ ]:
best_configs = {'HC': hc_base['best_state'], 'SA': sa_base['best_state'], 'GA': ga_base['best_state']}
fig, ax = plt.subplots(figsize=(10, 5)); quarters = range(1, T+1)
for algo, state in best_configs.items():
    ax.plot(quarters, state, 'o-', lw=2, ms=6, label=algo)
ax.axhline(y=params['r0'], color='gray', ls=':', alpha=0.5, label=f'r0 = {params["r0"]}%')
ax.fill_between(quarters, params['r_min'], params['r_max'], alpha=0.05, color='green')
ax.set_xlabel('Quarter'); ax.set_ylabel('BI-Rate (%)')
ax.set_title('Lintasan BI-Rate Optimal per Algoritma')
ax.set_xticks(quarters); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('best_trajectory.png', dpi=150); plt.show()

print('Best states:')
for algo, state in best_configs.items():
    print(f'  {algo}: {state}')

## 9. Kesimpulan & Pembelajaran

### Perjalanan Eksperimentasi

| Iterasi | Perubahan | Masalah Terungkap | Solusi |
|---------|-----------|-------------------|--------|
| 1 | Hapus C3 (IR differential) | Terlalu membatasi, bukan hard constraint riil | Model 5 constraint |
| 2 | Hapus C2 (smoothing hard) | Semua konvergen ke r_min=3.0%, C4 selalu violasi | — |
| 3 | Diagnosa model | $\rho_\pi=0.6$ -> ekonomi deflationary | $\rho_\pi=0.85$, $\phi=0.30$ |
| 4 | Model terkalibrasi | C4 feasible, tapi landscape cembung | Perlu ruggedness |
| 5 | Tambah C5 (penalized) | Landscape rugged, algoritma bersaing | **Final spec** |

### Konfigurasi Final

- **5 constraint**: C1 (hard), C2-C5 (penalized, $\mu=100$)
- **Parameter**: $\rho_\pi=0.85$, $\phi=0.30$, $T=8$, seed=42
- **State space**: $21^8 = 3.78 \times 10^{10}$
- **$r_t \in [3.00\%, 8.00\%]$**, kelipatan 25 bps
- **Initial state**: random uniform $\in [r_{min}, r_{max}]$

### Hasil Baseline (seed=42)

| Algo | J(s) | RMSD | pi_T | Feasible | C5 Viol |
|------|------|------|------|----------|---------|
| HC | -18.28 | 1.06 | 1.72% | NO | 0.25 |
| SA | -18.28 | 1.06 | 1.72% | NO | 0.25 |
| GA | -34.56 | 0.77 | 2.01% | NO | 0.50 |

### Temuan Kunci

1. **C5 menciptakan trade-off asli**: easing agresif = output **meningkat** namun penalti terhadap C5 vs easing gradual = output **meningkat** de C5 terpenuhi
2. **SA menunjukkan eksplorasi nyata**: J(s) ber-osilasi antara -35 dan -18 — bukti escape mechanism bekerja
3. **GA lebih konsisten**: RMSD lebih rendah (0.77 vs 1.06), pi_T lebih dekat target
4. **HC rentan local optima**: single run bisa terjebak di J=-65 (vs J=-18 untuk multi-restart)
5. **Robustness**: variasi signifikan di J(s) antar seed — landscape benar-benar rugged, tidak ada pemenang tunggal

### Rekomendasi

1. Gunakan **multi-restart HC** atau **SA** untuk eksplorasi lebih baik pada landscape rugged
2. **GA dengan diversity maintenance** untuk mencegah konvergensi prematur
3. Jika prioritas inflasi absolut, naikkan $w_\pi$ atau $\mu$ C4
4. Jika prioritas pertumbuhan (P2SK), biarkan C5 menghukum easing agresif sebagai soft constraint